# Training speed test

Measures how fast the 124M best-design model trains on this GPU (tokens per second and peak memory), comparing
the current recipe with fused AdamW + TF32, `torch.compile`, and bigger micro-batches, at 256 and 512 tokens of
context. Nothing is saved: it's a measurement, about 15 minutes on an L4 or A100 (most of it preparing the data).

1. **Use a GPU**: *Runtime → Change runtime type* → L4 or A100 (Colab Pro).
2. **Run all**. The last cell prints a table; copy it, or leave the notebook open for Claude to read.


In [ ]:
# @title Get the code
import os, sys

REPO_URL = "https://github.com/a-ramakrishnan/tinystories-ARM-model.git"
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PLATFORM = "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local"

if IN_COLAB or IN_KAGGLE:
    # Kaggle keeps everything under /kaggle/working as the notebook's output, so the code and data go to /tmp
    # and only the final zip is saved there.
    CLONE_DIR = "/content/tinystories-ARM-model" if IN_COLAB else "/tmp/tinystories-ARM-model"
    if not os.path.exists(CLONE_DIR):
        !git clone -q {REPO_URL} {CLONE_DIR}
    else:
        !git -C {CLONE_DIR} pull -q   # already cloned in this session: get the latest code
    %cd {CLONE_DIR}
    !pip install -q -r requirements.txt

assert os.path.exists("full-model.py"), "Open this notebook from the tinystories-ARM-model folder."
PROJECT_DIR = os.getcwd()
OUTPUT_DIR = "/kaggle/working" if IN_KAGGLE else PROJECT_DIR   # where the results zip is written
print(f"Running on {PLATFORM}. Project folder: {PROJECT_DIR}")

In [ ]:
# @title Run the speed test
contexts = "256,512"  # @param {type:"string"}
steps = 60  # @param {type:"integer"}

import torch
import ab_test

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (switch the runtime to a GPU)")
root = ab_test.ensure_dataset("BaoMinhTran/mini-fineweb-edu", "small", "text")
for ctx in [int(c) for c in contexts.split(",") if c.strip()]:
    batches = "32,64,128" if ctx <= 256 else "16,32,64"   # the same tokens per step at 512
    !python speedtest.py --data {root} --block {ctx} --batches {batches} --steps {steps}
